# Week 22: Multimodal Foundation Models & Vision-Language Architectures
## Notebook 03: Audio Transformers & Whisper Architecture from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Raw audio waveforms are 1D arrays sampled at 16,000 to 44,100 samples per second:
- A 30-second audio clip contains **480,000 raw floating-point numbers**.
- Passing 480,000 raw samples directly into a Transformer self-attention layer requires $(480,000)^2$ operations, crashing GPU memory.

In 2022, Radford et al. (OpenAI) released **Whisper**:
1. Converts raw audio into an **80-channel Log-Mel Spectrogram** (a 2D time-frequency image representation of human auditory perception).
2. Employs 1D convolutions to downsample the time dimension by 2x.
3. Encodes audio via a standard Transformer Encoder.
4. An Autoregressive Transformer Decoder predicts multilingual transcriptions, timestamps, and language IDs!

---

### 2. Mathematical Foundations: Log-Mel Filterbanks
1. **Short-Time Fourier Transform (STFT):**
   Computes localized DFT over overlapping window frames (window = 25ms, hop = 10ms):
   $$X(t, f) = \left| \sum_{n=0}^{N-1} x[n + t \cdot H] \cdot w[n] \cdot e^{-i 2\pi f n / N} \right|^2$$

2. **Mel Scale Conversion:**
   Human pitch perception is logarithmic: we distinguish 100Hz vs 200Hz easily, but 10,000Hz vs 10,100Hz sounds identical:
   $$m = 2595 \cdot \log_{10}\left( 1 + \frac{f}{700} \right)$$

3. **Log Compression:** $S_{\text{mel}} = \log(\text{MelFilterbank} \cdot |X(t, f)|^2 + 1e-5)$.

In [ ]:
import numpy as np

# Step 1: Synthesize a 1-Second Audio Waveform (16 kHz sampling rate)
sample_rate = 16000
duration_sec = 0.5
t = np.linspace(0, duration_sec, int(sample_rate * duration_sec), endpoint=False)

# Voice chord: fundamental frequency (220 Hz) + harmonics (440 Hz, 880 Hz)
audio_waveform = (
    0.6 * np.sin(2 * np.pi * 220 * t) +
    0.3 * np.sin(2 * np.pi * 440 * t) +
    0.1 * np.sin(2 * np.pi * 880 * t)
)
print(f"Raw Audio Waveform: {len(audio_waveform)} samples ({duration_sec}s at {sample_rate}Hz)")

# Step 2: Implement Simplified Log-Mel Spectrogram Extraction
def compute_mock_mel_spectrogram(waveform, num_frames=50, num_mel_bins=8):
    """
    Computes time-frequency spectrogram grid: (num_mel_bins, num_frames)
    """
    frame_len = len(waveform) // num_frames
    spectrogram = np.zeros((num_mel_bins, num_frames))
    
    for frame_idx in range(num_frames):
        chunk = waveform[frame_idx * frame_len : (frame_idx + 1) * frame_len]
        # FFT magnitude
        fft_mag = np.abs(np.fft.rfft(chunk))
        # Bin frequencies into mel filters
        bin_size = len(fft_mag) // num_mel_bins
        for b in range(num_mel_bins):
            spectrogram[b, frame_idx] = np.mean(fft_mag[b * bin_size : (b + 1) * bin_size])
            
    # Log compression
    return np.log(spectrogram + 1e-5)

log_mel = compute_mock_mel_spectrogram(audio_waveform, num_frames=40, num_mel_bins=8)

print("\n--- Log-Mel Spectrogram Representation (Whisper Input) ---")
print(f"Shape: {log_mel.shape} (8 Mel Frequency Bins x 40 Temporal Frames)")
print("Compression Ratio: 8,000 raw samples -> 320 spectrogram cells (25x compression!)")
print(f"Low frequency energy (Bin 0): {np.mean(log_mel[0]):.2f} (Captures 220Hz harmonic!)")
print(f"High frequency energy (Bin 7): {np.mean(log_mel[7]):.2f}")

### 3. Step-by-Step Code Walkthrough
1. **Auditory Transformation:** Converts 8,000 high-frequency waveform samples into a 2D time-frequency grid.
2. **Frequency Binning:** The mel filterbank aggregates energy into human-perceptual bands.
3. **Audio-Language Bridge:** The resulting 2D matrix is treated like an image strip and processed with standard Transformer encoders.

---

### 4. Industry Applications & Trade-offs
- **Automated Speech Recognition (ASR):** Whisper powers speech-to-text transcription across 90+ languages.
- **Timestamp Alignment:** Uses cross-attention weights to produce word-level subtitle timestamps.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Whisper use Log-Mel spectrograms instead of raw 1D waveforms?**
> **Answer:** Raw audio waveforms contain hundreds of thousands of samples per second, making $O(N^2)$ Transformer attention computationally intractable. Log-Mel spectrograms compress audio by ~30x into perceptually meaningful frequency bands that model human cochlear frequency sensitivity.

**Q2: How does Whisper handle speech translation vs transcription in a single model?**
> **Answer:** Whisper uses special control prefix tokens. Prepending `<|startoftranscript|> <|en|> <|transcribe|>` instructs the decoder to transcribe in English; prepending `<|es|> <|translate|>` prompts the decoder to translate the spoken speech into Spanish directly.